In [1]:
import pandas as pd
import numpy as np
import time
import requests
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.live.nba.endpoints import scoreboard
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb
import pickle
import os
from datetime import datetime, timedelta
import pytz

### Get the last 5 years basketball game NBA

In [3]:
MODEL_FILE = 'nba_total_model.pkl'
HISTORY_FILE = 'nba_5years_history.csv'

all_df = []
for season in ['2021-22','2022-23','2023-24','2024-25','2025-26']:
    print(f"Fetching {season}...")
    df = leaguegamefinder.LeagueGameFinder(season_nullable=season).get_data_frames()[0]
    all_df.append(df)
    time.sleep(1.5)

games = pd.concat(all_df)
games['GAME_DATE'] = pd.to_datetime(games['GAME_DATE'])
games = games.sort_values(['GAME_DATE','GAME_ID'])

home = games[games['MATCHUP'].str.contains('vs.', na=False)].copy()
away = games[games['MATCHUP'].str.contains('@', na=False)].copy()
home = home.add_suffix('_HOME')
away = away.add_suffix('_AWAY')
merged = pd.merge(home, away, left_on='GAME_ID_HOME', right_on='GAME_ID_AWAY')
merged['TOTAL_SCORE'] = merged['PTS_HOME'] + merged['PTS_AWAY']
merged['GAME_DATE'] = merged['GAME_DATE_HOME']
merged = merged.sort_values('GAME_DATE').reset_index(drop=True)

merged['POSS_HOME'] = merged['FGA_HOME'] + merged['TOV_HOME'] + 0.44*merged['FTA_HOME'] - merged['OREB_HOME']
merged['POSS_AWAY'] = merged['FGA_AWAY'] + merged['TOV_AWAY'] + 0.44*merged['FTA_AWAY'] - merged['OREB_AWAY']
merged['GAME_PACE'] = (merged['POSS_HOME'] + merged['POSS_AWAY']) / 2

long = []
for _, r in merged.iterrows():
    long.append({'GAME_DATE': r['GAME_DATE'], 'TEAM_ABBREVIATION': r['TEAM_ABBREVIATION_HOME'],
                 'PTS': r['PTS_HOME'], 'OPP_PTS': r['PTS_AWAY'], 'PACE': r['GAME_PACE'],
                 'FG_PCT': r['FG_PCT_HOME'], 'GAME_ID': r['GAME_ID_HOME']})
    long.append({'GAME_DATE': r['GAME_DATE'], 'TEAM_ABBREVIATION': r['TEAM_ABBREVIATION_AWAY'],
                 'PTS': r['PTS_AWAY'], 'OPP_PTS': r['PTS_HOME'], 'PACE': r['GAME_PACE'],
                 'FG_PCT': r['FG_PCT_AWAY'], 'GAME_ID': r['GAME_ID_HOME']})
long_df = pd.DataFrame(long).sort_values(['TEAM_ABBREVIATION','GAME_DATE'])

# FIX: Do REST/B2B BEFORE saving, so history file complete
long_df['PREV_DATE'] = long_df.groupby('TEAM_ABBREVIATION')['GAME_DATE'].shift(1)
long_df['REST_DAYS'] = (long_df['GAME_DATE'] - long_df['PREV_DATE']).dt.days.fillna(3).clip(1, 3)
long_df['IS_B2B'] = (long_df['REST_DAYS'] == 1).astype(int)
long_df.to_csv(HISTORY_FILE, index=False)
print(f"Saved: {len(long_df)} rows")

for col in ['PTS','OPP_PTS','PACE','FG_PCT']:
    long_df[f'{col}_LAST5'] = long_df.groupby('TEAM_ABBREVIATION')[col].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
    long_df[f'{col}_SEASON'] = long_df.groupby('TEAM_ABBREVIATION')[col].transform(lambda x: x.shift(1).rolling(20, min_periods=1).mean())

long_df = long_df.drop_duplicates(subset=['GAME_ID', 'TEAM_ABBREVIATION'], keep='first')
feat_map = long_df.set_index(['GAME_ID','TEAM_ABBREVIATION']).to_dict('index')
def getf(gid, team, feat):
    try: return feat_map[(gid, team)][feat]
    except: return np.nan

rows = []
for _, r in merged.iterrows():
    f = {'TOTAL_SCORE': r['TOTAL_SCORE']}
    for col in ['PTS_LAST5','OPP_PTS_LAST5','PACE_LAST5','FG_PCT_LAST5','PTS_SEASON','OPP_PTS_SEASON','PACE_SEASON']:
        f[f'HOME_{col}'] = getf(r['GAME_ID_HOME'], r['TEAM_ABBREVIATION_HOME'], col)
        f[f'AWAY_{col}'] = getf(r['GAME_ID_HOME'], r['TEAM_ABBREVIATION_AWAY'], col)
    f['HOME_REST'] = getf(r['GAME_ID_HOME'], r['TEAM_ABBREVIATION_HOME'], 'REST_DAYS')
    f['AWAY_REST'] = getf(r['GAME_ID_HOME'], r['TEAM_ABBREVIATION_AWAY'], 'REST_DAYS')
    f['HOME_B2B'] = getf(r['GAME_ID_HOME'], r['TEAM_ABBREVIATION_HOME'], 'IS_B2B')
    f['AWAY_B2B'] = getf(r['GAME_ID_HOME'], r['TEAM_ABBREVIATION_AWAY'], 'IS_B2B')
    f['EXPECTED_TOTAL'] = f['HOME_PTS_LAST5'] + f['AWAY_PTS_LAST5']
    f['PACE_AVG'] = (f['HOME_PACE_LAST5'] + f['AWAY_PACE_LAST5'])/2
    f['DEF_AVG'] = (f['HOME_OPP_PTS_LAST5'] + f['AWAY_OPP_PTS_LAST5'])/2
    rows.append(f)

df_final = pd.DataFrame(rows).dropna()
feature_cols = ['HOME_PTS_LAST5','AWAY_PTS_LAST5','HOME_OPP_PTS_LAST5','AWAY_OPP_PTS_LAST5',
                'HOME_PACE_LAST5','AWAY_PACE_LAST5','HOME_FG_PCT_LAST5','AWAY_FG_PCT_LAST5',
                'HOME_PTS_SEASON','AWAY_PTS_SEASON','EXPECTED_TOTAL','PACE_AVG','DEF_AVG',
                'HOME_REST','AWAY_REST','HOME_B2B','AWAY_B2B']
X = df_final[feature_cols]
y = df_final['TOTAL_SCORE']
split = int(len(df_final)*0.85)
model = xgb.XGBRegressor(n_estimators=800, max_depth=5, learning_rate=0.03, subsample=0.8, random_state=42, n_jobs=-1)
model.fit(X.iloc[:split], y.iloc[:split])
pred = model.predict(X.iloc[split:])
print(f"MAE: {mean_absolute_error(y.iloc[split:], pred):.2f} | Within 5: {(abs(y.iloc[split:]-pred)<=5).mean():.1%} | Pred {pred.min():.1f}-{pred.max():.1f}")
pickle.dump(model, open(MODEL_FILE,'wb'))
pickle.dump(feature_cols, open('feature_cols.pkl','wb'))
print("Model saved ✅")

Fetching 2021-22...
Fetching 2022-23...
Fetching 2023-24...
Fetching 2024-25...
Fetching 2025-26...
Saved: 13946 rows
MAE: 16.01 | Within 5: 19.0% | Pred 47.3-254.3
Model saved ✅


In [4]:
HISTORY_FILE = 'nba_5years_history.csv'

print("Fetching only 2026-27...")
df_new = leaguegamefinder.LeagueGameFinder(season_nullable='2026-27').get_data_frames()[0]
df_new['GAME_DATE'] = pd.to_datetime(df_new['GAME_DATE'])
df_new = df_new.sort_values(['GAME_DATE','GAME_ID'])

home = df_new[df_new['MATCHUP'].str.contains('vs.', na=False)].copy()
away = df_new[df_new['MATCHUP'].str.contains('@', na=False)].copy()
home = home.add_suffix('_HOME'); away = away.add_suffix('_AWAY')
merged = pd.merge(home, away, left_on='GAME_ID_HOME', right_on='GAME_ID_AWAY')
merged['GAME_DATE'] = merged['GAME_DATE_HOME']
merged['POSS_HOME'] = merged['FGA_HOME'] + merged['TOV_HOME'] + 0.44*merged['FTA_HOME'] - merged['OREB_HOME']
merged['POSS_AWAY'] = merged['FGA_AWAY'] + merged['TOV_AWAY'] + 0.44*merged['FTA_AWAY'] - merged['OREB_AWAY']
merged['GAME_PACE'] = (merged['POSS_HOME'] + merged['POSS_AWAY']) / 2

long_new = []
for _, r in merged.iterrows():
    long_new.append({'GAME_DATE': r['GAME_DATE'], 'TEAM_ABBREVIATION': r['TEAM_ABBREVIATION_HOME'], 'PTS': r['PTS_HOME'], 'OPP_PTS': r['PTS_AWAY'], 'PACE': r['GAME_PACE'], 'FG_PCT': r['FG_PCT_HOME'], 'GAME_ID': r['GAME_ID_HOME']})
    long_new.append({'GAME_DATE': r['GAME_DATE'], 'TEAM_ABBREVIATION': r['TEAM_ABBREVIATION_AWAY'], 'PTS': r['PTS_AWAY'], 'OPP_PTS': r['PTS_HOME'], 'PACE': r['GAME_PACE'], 'FG_PCT': r['FG_PCT_AWAY'], 'GAME_ID': r['GAME_ID_HOME']})
long_new = pd.DataFrame(long_new)

old = pd.read_csv(HISTORY_FILE, parse_dates=['GAME_DATE'])
# Keep only old columns, drop any raw columns if corrupted
old = old[['GAME_DATE','TEAM_ABBREVIATION','PTS','OPP_PTS','PACE','FG_PCT','GAME_ID']]
combined = pd.concat([old, long_new], ignore_index=True)
combined = combined.drop_duplicates(subset=['GAME_ID','TEAM_ABBREVIATION'], keep='last')
combined = combined.sort_values(['TEAM_ABBREVIATION','GAME_DATE'])
combined['PREV_DATE'] = combined.groupby('TEAM_ABBREVIATION')['GAME_DATE'].shift(1)
combined['REST_DAYS'] = (combined['GAME_DATE'] - combined['PREV_DATE']).dt.days.fillna(3).clip(1, 3)
combined['IS_B2B'] = (combined['REST_DAYS'] == 1).astype(int)
combined.to_csv(HISTORY_FILE, index=False)
print(f"✅ Total: {len(combined)} | Max date: {combined['GAME_DATE'].max()}")

Fetching only 2026-27...
✅ Total: 14406 | Max date: 2026-11-11 00:00:00


In [5]:
INJURY_CACHE = {}
INJURY_CACHE_TIME = 0
CACHE_TTL = 3600

TEAM_NAME_TO_ABBR = {
    'Atlanta Hawks':'ATL','Boston Celtics':'BOS','Brooklyn Nets':'BKN','Charlotte Hornets':'CHA',
    'Chicago Bulls':'CHI','Cleveland Cavaliers':'CLE','Dallas Mavericks':'DAL','Denver Nuggets':'DEN',
    'Detroit Pistons':'DET','Golden State Warriors':'GSW','Houston Rockets':'HOU','Indiana Pacers':'IND',
    'LA Clippers':'LAC','Los Angeles Lakers':'LAL','Memphis Grizzlies':'MEM','Miami Heat':'MIA',
    'Milwaukee Bucks':'MIL','Minnesota Timberwolves':'MIN','New Orleans Pelicans':'NOP','New York Knicks':'NYK',
    'Oklahoma City Thunder':'OKC','Orlando Magic':'ORL','Philadelphia 76ers':'PHI','Phoenix Suns':'PHX',
    'Portland Trail Blazers':'POR','Sacramento Kings':'SAC','San Antonio Spurs':'SAS','Toronto Raptors':'TOR',
    'Utah Jazz':'UTA','Washington Wizards':'WAS'
}
STARS = {'Anthony Edwards','Paolo Banchero','Jimmy Butler','Jalen Suggs','Trae Young','Joel Embiid','Tyrese Maxey','Luka Doncic','LeBron James','Stephen Curry','Kevin Durant','Jayson Tatum','Jaylen Brown','Nikola Jokic','Giannis Antetokounmpo','Shai Gilgeous-Alexander','Anthony Davis','Ja Morant','Damian Lillard','Devin Booker','Kyrie Irving','Jalen Johnson','Jalen Brunson','Tyrese Haliburton','Donovan Mitchell','Zion Williamson','Chet Holmgren','Desmond Bane','Jalen Williams'}

def load_injuries_real(force_refresh=False):
    global INJURY_CACHE, INJURY_CACHE_TIME
    if INJURY_CACHE and not force_refresh and (time.time() - INJURY_CACHE_TIME) < CACHE_TTL:
        return INJURY_CACHE
    INJURY_CACHE = {}
    try:
        data = requests.get("https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries", timeout=10).json()
        for team_entry in data.get('injuries', []):
            abbr = TEAM_NAME_TO_ABBR.get(team_entry.get('displayName',''))
            if not abbr: continue
            outs = []; out_count = 0
            for inj in team_entry.get('injuries', []):
                player = inj.get('athlete', {}).get('displayName','?')
                status = inj.get('status','')
                if status in ['Out','Out for Season','Out Indefinitely','Out For Season']:
                    outs.append(f"{player} (Out)"); out_count += 1
                elif status in ['Doubtful','Questionable','Day-To-Day']:
                    outs.append(f"{player} ({status})")
            INJURY_CACHE[abbr] = (outs, out_count)
        INJURY_CACHE_TIME = time.time()
        print(f"Injuries loaded for {len(INJURY_CACHE)} teams ✅")
    except Exception as e:
        print(f"Injury fetch fail: {e}")
    return INJURY_CACHE

def get_injuries_live(team_abbr):
    cache = load_injuries_real()
    if team_abbr not in cache: return "Healthy", 0, 0.0
    outs, out_count = cache[team_abbr]
    if not outs: return "Healthy", 0, 0.0
    adj = 0.0
    for o in outs:
        name = o.split(' (')[0]
        if '(Out)' in o:
            adj += -4.0 if name in STARS else -0.5
    adj = max(adj, -8.0)
    return "OUT: " + ", ".join(outs[:4]), out_count, adj

In [7]:
# === AFTER you fetch games_today, ADD THIS ===
# No MY_LINES needed again. We go ask you one by one.

MY_LINES = {}  # empty first

print("=== TYPE YOUR BET9JA LINE FOR EACH GAME ===")
print("If you no wan bet that game, just press Enter (leave blank)\n")

for g in games_today:
    home_abbr, away_abbr = g['home'], g['away']
    key = f"{away_abbr}@{home_abbr}"  # same format as before: AWAY@HOME
    display = f"{away_abbr} @ {home_abbr}"
    try:
        user_line = input(f"Bet MAIN LINE for {display} (e.g 226.5) or Enter to skip: ").strip()
        if user_line == "":
            print(f"Skipped {display}\n")
            continue
        MY_LINES[key] = float(user_line)
        print(f"Saved {display} = {user_line}\n")
    except ValueError:
        print("Invalid number, skipped\n")

print(f"Lines you enter: {MY_LINES}\n")
print("Now e go predict...\n")

# === THEN YOUR NORMAL PREDICT LOOP STARTS HERE ===
for g in games_today:
    home_abbr, away_abbr = g['home'], g['away']
    key = f"{away_abbr}@{home_abbr}"
    print(f"--- {away_abbr} @ {home_abbr} | {g['status']} ---")
    home_inj, home_out, home_adj = get_injuries_live(home_abbr)
    away_inj, away_out, away_adj = get_injuries_live(away_abbr)
    print(f"HOME Inj: {home_inj} (OUT: {home_out}) | AWAY Inj: {away_inj} (OUT: {away_out})")
    hf = get_team_live_features(home_abbr, history); af = get_team_live_features(away_abbr, history)
    print(f"HOME Last5: {hf['PTS_LAST5']:.1f} | AWAY Last5: {af['PTS_LAST5']:.1f} | Rest: {hf['REST']}/{af['REST']} | Adj: {home_adj:.1f}/{away_adj:.1f}")

    feat = {'HOME_PTS_LAST5': hf['PTS_LAST5'],'AWAY_PTS_LAST5': af['PTS_LAST5'],'HOME_OPP_PTS_LAST5': hf['OPP_PTS_LAST5'],'AWAY_OPP_PTS_LAST5': af['OPP_PTS_LAST5'],'HOME_PACE_LAST5': hf['PACE_LAST5'],'AWAY_PACE_LAST5': af['PACE_LAST5'],'HOME_FG_PCT_LAST5': hf['FG_PCT_LAST5'],'AWAY_FG_PCT_LAST5': af['FG_PCT_LAST5'],'HOME_PTS_SEASON': hf['PTS_SEASON'],'AWAY_PTS_SEASON': af['PTS_SEASON'],'EXPECTED_TOTAL': hf['PTS_LAST5']+af['PTS_LAST5'],'PACE_AVG': (hf['PACE_LAST5']+af['PACE_LAST5'])/2,'DEF_AVG': (hf['OPP_PTS_LAST5']+af['OPP_PTS_LAST5'])/2,'HOME_REST': hf['REST'],'AWAY_REST': af['REST'],'HOME_B2B': hf['B2B'],'AWAY_B2B': af['B2B']}
    base_pred = float(model.predict(pd.DataFrame([feat])[feature_cols])[0])
    pred_total = base_pred + home_adj + away_adj
    if pred_total < 180 or pred_total > 260:
        print(f"⚠️ Model gave {pred_total:.1f} - using EXPECTED_TOTAL")
        pred_total = feat['EXPECTED_TOTAL'] + home_adj + away_adj
    print(f">>> PREDICTED TOTAL: {pred_total:.1f} (Base {base_pred:.1f} + Adj {home_adj+away_adj:.1f})")
    if key in MY_LINES:
        line = MY_LINES[key]; edge = pred_total - line; pick = "OVER" if edge>0 else "UNDER"
        print(f"Bookie: {line} | Edge: {edge:+.1f} | Pick: {pick}")
        if abs(edge)>=7: print("✅ STRONG BET - 1-2% bankroll")
        elif abs(edge)>=2: print("⚠️ SMALL EDGE - Skip or small")
        else: print("❌ NO BET - SKIP")
    else: print(f"No line for {key} - you skip am")
    print()

=== TYPE YOUR BET9JA LINE FOR EACH GAME ===
If you no wan bet that game, just press Enter (leave blank)



Bet MAIN LINE for TOR @ PHI (e.g 226.5) or Enter to skip:  225


Saved TOR @ PHI = 225



Bet MAIN LINE for DET @ WAS (e.g 226.5) or Enter to skip:  220


Saved DET @ WAS = 220



Bet MAIN LINE for DAL @ MIN (e.g 226.5) or Enter to skip:  300


... (17 outputs left out)

Bet MAIN LINE for POR @ SAC (e.g 226.5) or Enter to skip:  200


Saved POR @ SAC = 200



Bet MAIN LINE for OKC @ LAC (e.g 226.5) or Enter to skip:  200


Saved OKC @ LAC = 200

Lines you enter: {'TOR@PHI': 225.0, 'DET@WAS': 220.0, 'DAL@MIN': 300.0, 'CHA@PHX': 300.0, 'NYK@IND': 215.0, 'MIL@ORL': 200.0, 'CLE@MIA': 200.0, 'ATL@CHI': 200.0, 'MEM@NOP': 200.0, 'UTA@HOU': 200.0, 'GSW@SAS': 200.0, 'POR@SAC': 200.0, 'OKC@LAC': 200.0}

Now e go predict...

--- TOR @ PHI | Scheduled ---
HOME Inj: OUT: LeBron James (Out), Dominick Barlow (Out), Adem Bona (Out), Joel Embiid (Day-To-Day) (OUT: 4) | AWAY Inj: OUT: Trayce Jackson-Davis (Out), Scottie Barnes (Out), Kawhi Leonard (Out), Nate Bittle (Out) (OUT: 4)
HOME Last5: 112.2 | AWAY Last5: 110.0 | Rest: 2/1 | Adj: -5.5/-2.0
>>> PREDICTED TOTAL: 221.6 (Base 229.1 + Adj -7.5)
Bookie: 225.0 | Edge: -3.4 | Pick: UNDER
⚠️ SMALL EDGE - Skip or small

--- DET @ WAS | Scheduled ---
HOME Inj: OUT: Tristan Vukcevic (Out), Reece Beekman (Out), Justin Champagnie (Out), Jamir Watkins (Out) (OUT: 4) | AWAY Inj: OUT: Ugonna Onyenso (Day-To-Day), Isaiah Joe (Out), Jalen Duren (Out) (OUT: 2)
HOME Last5: 104.8 | AWAY

In [10]:
# Save model
with open('nba_total_model.pkl','wb') as f:
    pickle.dump(model, f)

# Save feature columns - VERY IMPORTANT, must be same order for prediction
with open('feature_cols.pkl','wb') as f:
    pickle.dump(feature_cols, f)

print("Saved!")
print("Model:", 'nba_total_model.pkl')
print("Features:", feature_cols)

Saved!
Model: nba_total_model.pkl
Features: ['HOME_PTS_LAST5', 'AWAY_PTS_LAST5', 'HOME_OPP_PTS_LAST5', 'AWAY_OPP_PTS_LAST5', 'HOME_PACE_LAST5', 'AWAY_PACE_LAST5', 'HOME_FG_PCT_LAST5', 'AWAY_FG_PCT_LAST5', 'HOME_PTS_SEASON', 'AWAY_PTS_SEASON', 'EXPECTED_TOTAL', 'PACE_AVG', 'DEF_AVG', 'HOME_REST', 'AWAY_REST', 'HOME_B2B', 'AWAY_B2B']
